# 03 Data Cleaning - Cleaning Terpisah per Dataset

Notebook ini membersihkan 8 dataset secara terpisah berdasarkan output gathering dan assessment. Setiap dataset menghasilkan file cleaned sendiri di `data/interim/cleaned_separate/`.

Output utama notebook ini:
- `data/interim/cleaned_separate/<dataset_id>_cleaned.csv`
- `data/interim/cleaning_reports_by_dataset.json`
- `data/interim/cleaning_summary_by_dataset.csv`
- `reports/03_cleaning_report_by_dataset.txt`
- visualisasi akhir dari summary cleaned per dataset.

## Setup Environment dan Load Input

In [1]:
import json
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, HTML

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

try:
    from fingo_ds1.config import INTERIM_DATA_PATH, REPORTS_PATH
except ImportError:
    INTERIM_DATA_PATH = project_root / 'data' / 'interim'
    REPORTS_PATH = project_root / 'reports'

interim_data_path = Path(INTERIM_DATA_PATH)
reports_path = Path(REPORTS_PATH)
raw_separate_path = interim_data_path / 'raw_separate'
cleaned_separate_path = interim_data_path / 'cleaned_separate'
figures_path = reports_path / 'figures'

cleaned_separate_path.mkdir(parents=True, exist_ok=True)
figures_path.mkdir(parents=True, exist_ok=True)
reports_path.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

manifest_path = raw_separate_path / 'raw_dataset_manifest.json'
assessment_path = interim_data_path / 'assessment_reports_by_dataset.json'

if not manifest_path.exists():
    raise FileNotFoundError('Manifest raw dataset tidak ditemukan. Jalankan 01_data_gathering.ipynb terlebih dahulu.')
if not assessment_path.exists():
    raise FileNotFoundError('Assessment report tidak ditemukan. Jalankan 02_data_assessing.ipynb terlebih dahulu.')

with open(manifest_path, 'r', encoding='utf-8') as file:
    dataset_catalog = json.load(file)
with open(assessment_path, 'r', encoding='utf-8') as file:
    assessment_reports = json.load(file)

raw_datasets = {}
for meta in dataset_catalog:
    csv_path = project_root / meta['raw_output_path']
    raw_datasets[meta['dataset_id']] = pd.read_csv(csv_path, low_memory=False)
    print(f"Loaded {meta['dataset_id']}: {raw_datasets[meta['dataset_id']].shape}")

Loaded ecommerce_2024_01_january: (431, 24)
Loaded ecommerce_2024_06_june: (697, 24)
Loaded ecommerce_2024_12_december: (1214, 23)
Loaded ecommerce_2025_02_february: (957, 24)
Loaded ecommerce_2025_07_july: (766, 23)
Loaded ecommerce_2025_11_november: (1131, 24)
Loaded daily_household_transactions: (2461, 14)
Loaded personal_finance: (1500, 11)


## Helper Cleaning
Helper ini melakukan parsing tanggal dan amount, standarisasi teks, imputasi ringan, drop record invalid yang kritis, capping outlier amount per dataset, dan pembuatan schema cleaned.

In [2]:
COLUMN_CANDIDATES = {
    'date': ['date', 'waktu_pesanan_dibuat', 'transaction_date', 'order_date'],
    'amount': ['amount', 'total_pembayaran', 'total_payment', 'payment_amount'],
    'category': ['category', 'product_categories', 'product_category', 'subcategory'],
    'description': ['transaction_description', 'description', 'note', 'order_id'],
    'transaction_type': ['type', 'income_expense', 'income/expense', 'status_pesanan'],
    'status': ['status_pesanan'],
    'payment_method': ['metode_pembayaran', 'mode', 'payment_method'],
    'city': ['kota_kabupaten', 'city'],
    'province': ['provinsi', 'province'],
    'quantity': ['total_qty', 'quantity'],
    'order_id': ['order_id'],
}


def normalize_name(value):
    text = str(value).strip().lower()
    text = re.sub(r'[/\\]+', '_', text)
    text = re.sub(r'[^0-9a-zA-Z]+', '_', text)
    return text.strip('_')


def find_column(frame, candidates):
    lookup = {normalize_name(column): column for column in frame.columns}
    normalized_candidates = [normalize_name(candidate) for candidate in candidates]

    for candidate in normalized_candidates:
        if candidate in lookup:
            return lookup[candidate]

    for candidate in normalized_candidates:
        for normalized_column, original_column in lookup.items():
            if candidate and candidate in normalized_column:
                return original_column

    return None


def detect_columns(frame):
    return {name: find_column(frame, candidates) for name, candidates in COLUMN_CANDIDATES.items()}


def parse_mixed_dates(series):
    values = series.astype('string').str.strip().replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})

    try:
        parsed = pd.to_datetime(values, errors='coerce', format='mixed', dayfirst=True)
    except TypeError:
        parsed = pd.to_datetime(values, errors='coerce', dayfirst=True)

    numeric_values = pd.to_numeric(values, errors='coerce')
    excel_serial_mask = parsed.isna() & numeric_values.between(20000, 80000)
    if excel_serial_mask.any():
        parsed.loc[excel_serial_mask] = pd.to_datetime(
            numeric_values.loc[excel_serial_mask], unit='D', origin='1899-12-30', errors='coerce'
        )

    return parsed


def parse_amount(series):
    values = series.astype('string').str.strip()
    values = values.str.replace(r'\s+', '', regex=True)
    values = values.str.replace(r'(?<=\d),(?=\d{3}(\D|$))', '', regex=True)
    values = values.str.replace(',', '.', regex=False)
    values = values.str.replace(r'[^0-9.\-]', '', regex=True)
    values = values.replace({'': np.nan, 'nan': np.nan, 'None': np.nan, '<NA>': np.nan})
    return pd.to_numeric(values, errors='coerce')


def clean_text(series, fill_value='unknown'):
    return (
        series.fillna(fill_value)
        .astype('string')
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .replace('', fill_value)
    )


def clean_category(series):
    cleaned = clean_text(series, 'unknown').str.lower()
    cleaned = cleaned.str.replace('&', ' and ', regex=False)
    cleaned = cleaned.str.replace(r'[^0-9a-zA-Z]+', '_', regex=True)
    cleaned = cleaned.str.strip('_').replace('', 'unknown')

    category_mapping = {
        'food_drink': 'food_and_drink',
        'food_and_drink': 'food_and_drink',
        'transport': 'transportation',
        'subscription': 'subscription',
        'rent': 'rent',
        'utilities': 'utilities',
    }
    return cleaned.replace(category_mapping)


def clean_optional_text(frame, column, fill_value='unknown'):
    if column and column in frame.columns:
        return clean_text(frame[column], fill_value)
    return pd.Series(fill_value, index=frame.index, dtype='string')


def clean_optional_number(frame, column):
    if column and column in frame.columns:
        return parse_amount(frame[column])
    return pd.Series(np.nan, index=frame.index)


def date_from_period(meta, index):
    period = meta.get('dataset_period')
    if not period:
        return pd.Series(pd.NaT, index=index)
    return pd.Series(pd.to_datetime(f'{period}-01', errors='coerce'), index=index)


def to_python(value):
    if isinstance(value, dict):
        return {key: to_python(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_python(item) for item in value]
    if isinstance(value, tuple):
        return [to_python(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if np.isnan(value) else float(value)
    if isinstance(value, (pd.Timestamp, datetime)):
        return None if pd.isna(value) else value.isoformat()
    if pd.isna(value):
        return None
    return value


def log_cleaning_step(log, step_name, before_count, after_count, details=None):
    log.append(
        {
            'step': step_name,
            'before_records': int(before_count),
            'after_records': int(after_count),
            'removed_records': int(before_count - after_count),
            'details': details or {},
        }
    )

## Fungsi Cleaning per Dataset

In [3]:
def clean_dataset(dataset_id, frame, meta, assessment):
    working = frame.copy()
    initial_records = len(working)
    cleaning_log = []

    detected = assessment.get('schema', {}).get('detected_columns') or detect_columns(working)
    date_column = detected.get('date')
    amount_column = detected.get('amount')
    category_column = detected.get('category')
    description_column = detected.get('description')

    if date_column and date_column in working.columns:
        working['_date_clean'] = parse_mixed_dates(working[date_column])
        date_source = 'source_column'
    else:
        working['_date_clean'] = date_from_period(meta, working.index)
        date_source = 'dataset_period_from_filename' if meta.get('dataset_period') else 'not_available'

    before = len(working)
    working = working.loc[working['_date_clean'].notna()].copy()
    log_cleaning_step(cleaning_log, 'Remove rows without valid date', before, len(working), {'date_source': date_source})

    if amount_column and amount_column in working.columns:
        working['_amount_original'] = parse_amount(working[amount_column]).astype(float)
    else:
        working['_amount_original'] = np.nan

    before = len(working)
    working = working.loc[working['_amount_original'].notna()].copy()
    working = working.loc[working['_amount_original'] >= 0].copy()
    log_cleaning_step(cleaning_log, 'Remove rows without valid non-negative amount', before, len(working), {'amount_column': amount_column})

    cap_value = np.nan
    capped_records = 0
    working['_amount_clean'] = working['_amount_original'].astype(float)
    positive_amounts = working.loc[working['_amount_original'] > 0, '_amount_original']
    if len(positive_amounts) >= 10:
        cap_value = float(positive_amounts.quantile(0.99))
        capped_records = int((working['_amount_original'] > cap_value).sum())
        working['_amount_clean'] = working['_amount_original'].astype(float).clip(upper=cap_value)
    log_cleaning_step(
        cleaning_log,
        'Cap high amount outliers at p99 per dataset',
        len(working),
        len(working),
        {'cap_value': cap_value, 'capped_records': capped_records},
    )

    category_source = working[category_column] if category_column and category_column in working.columns else pd.Series('unknown', index=working.index)
    working['_category_clean'] = clean_category(category_source)

    description_source = working[description_column] if description_column and description_column in working.columns else pd.Series('', index=working.index)
    working['_description_clean'] = clean_text(description_source, '')

    standard_frame = pd.DataFrame(
        {
            'dataset_id': dataset_id,
            'dataset_name': meta['dataset_name'],
            'domain': meta['domain'],
            'source_file': clean_optional_text(working, '_source_file', meta.get('source_path', 'unknown')),
            'dataset_period': meta.get('dataset_period'),
            'date': working['_date_clean'],
            'amount': working['_amount_clean'].astype(float),
            'amount_original': working['_amount_original'].astype(float),
            'category': working['_category_clean'],
            'description': working['_description_clean'],
            'transaction_type': clean_optional_text(working, detected.get('transaction_type'), 'unknown'),
            'status': clean_optional_text(working, detected.get('status'), 'unknown'),
            'payment_method': clean_optional_text(working, detected.get('payment_method'), 'unknown'),
            'city': clean_optional_text(working, detected.get('city'), 'unknown'),
            'province': clean_optional_text(working, detected.get('province'), 'unknown'),
            'quantity': clean_optional_number(working, detected.get('quantity')),
            'order_id': clean_optional_text(working, detected.get('order_id'), ''),
            'date_source': date_source,
            'amount_cap_p99': cap_value,
        }
    )

    text_columns = ['category', 'description', 'transaction_type', 'status', 'payment_method', 'city', 'province', 'order_id']
    for column in text_columns:
        standard_frame[column] = clean_text(standard_frame[column], 'unknown' if column != 'description' else '')

    standard_frame['month'] = pd.to_datetime(standard_frame['date']).dt.to_period('M').astype(str)
    standard_frame['year'] = pd.to_datetime(standard_frame['date']).dt.year.astype('Int64')

    before = len(standard_frame)
    standard_frame = standard_frame.drop_duplicates(
        subset=['date', 'amount', 'category', 'description', 'transaction_type'], keep='first'
    ).reset_index(drop=True)
    log_cleaning_step(cleaning_log, 'Remove duplicate cleaned transaction keys', before, len(standard_frame))

    summary = {
        'dataset_id': dataset_id,
        'dataset_name': meta['dataset_name'],
        'domain': meta['domain'],
        'initial_records': int(initial_records),
        'final_records': int(len(standard_frame)),
        'removed_records': int(initial_records - len(standard_frame)),
        'retention_pct': round(float(len(standard_frame) / initial_records * 100), 2) if initial_records else 0.0,
        'date_source': date_source,
        'amount_column': amount_column,
        'category_column': category_column,
        'capped_records': capped_records,
        'amount_cap_p99': cap_value,
        'missing_cells_after': int(standard_frame.isna().sum().sum()),
        'duplicate_keys_after': int(standard_frame.duplicated(subset=['date', 'amount', 'category', 'description', 'transaction_type']).sum()),
        'total_amount': float(standard_frame['amount'].sum()) if len(standard_frame) else 0.0,
        'median_amount': float(standard_frame['amount'].median()) if len(standard_frame) else 0.0,
        'top_category': standard_frame['category'].mode().iloc[0] if len(standard_frame) and not standard_frame['category'].mode().empty else None,
        'cleaning_steps': cleaning_log,
    }

    return standard_frame, summary

## Jalankan Cleaning dan Simpan Output Terpisah

In [4]:
cleaned_datasets = {}
cleaning_reports = {}
summary_rows = []

for meta in dataset_catalog:
    dataset_id = meta['dataset_id']
    cleaned_frame, summary = clean_dataset(dataset_id, raw_datasets[dataset_id], meta, assessment_reports[dataset_id])

    output_path = cleaned_separate_path / f'{dataset_id}_cleaned.csv'
    cleaned_frame.to_csv(output_path, index=False)

    summary['output_path'] = str(output_path.relative_to(project_root))
    cleaned_datasets[dataset_id] = cleaned_frame
    cleaning_reports[dataset_id] = summary
    summary_rows.append({key: value for key, value in summary.items() if key != 'cleaning_steps'})

    print(f"{dataset_id}: {summary['initial_records']:,} -> {summary['final_records']:,} rows | saved {output_path.relative_to(project_root)}")

cleaning_summary_df = pd.DataFrame(summary_rows).sort_values('dataset_id').reset_index(drop=True)
display(cleaning_summary_df)

ecommerce_2024_01_january: 431 -> 431 rows | saved data/interim/cleaned_separate/ecommerce_2024_01_january_cleaned.csv
ecommerce_2024_06_june: 697 -> 697 rows | saved data/interim/cleaned_separate/ecommerce_2024_06_june_cleaned.csv
ecommerce_2024_12_december: 1,214 -> 1,214 rows | saved data/interim/cleaned_separate/ecommerce_2024_12_december_cleaned.csv
ecommerce_2025_02_february: 957 -> 957 rows | saved data/interim/cleaned_separate/ecommerce_2025_02_february_cleaned.csv
ecommerce_2025_07_july: 766 -> 766 rows | saved data/interim/cleaned_separate/ecommerce_2025_07_july_cleaned.csv
ecommerce_2025_11_november: 1,131 -> 1,131 rows | saved data/interim/cleaned_separate/ecommerce_2025_11_november_cleaned.csv
daily_household_transactions: 2,461 -> 2,445 rows | saved data/interim/cleaned_separate/daily_household_transactions_cleaned.csv
personal_finance: 1,500 -> 1,500 rows | saved data/interim/cleaned_separate/personal_finance_cleaned.csv


,dataset_id,dataset_name,domain,initial_records,final_records,removed_records,retention_pct,date_source,amount_column,category_column,capped_records,amount_cap_p99,missing_cells_after,duplicate_keys_after,total_amount,median_amount,top_category,output_path
0,daily_household_transactions,Daily Household Transactions,household_finance,2461,2445,16,99.35,source_column,Amount,Category,25,62292.80,4890,0,5898107.98,100.000,food,data/interim/cleaned_separate/daily_household_...
1,ecommerce_2024_01_january,E-Commerce Sales - January 2024,ecommerce_sales,431,431,0,100.00,source_column,Total Pembayaran,product_categories,4,703122.34,0,0,23760242.36,30000.000,celengan,data/interim/cleaned_separate/ecommerce_2024_0...
2,ecommerce_2024_06_june,E-Commerce Sales - June 2024,ecommerce_sales,697,697,0,100.00,source_column,Total Pembayaran,product_categories,6,538000.00,0,0,30399557.00,21696.000,celengan,data/interim/cleaned_separate/ecommerce_2024_0...
3,ecommerce_2024_12_december,E-Commerce Sales - December 2024,ecommerce_sales,1214,1214,0,100.00,dataset_period_from_filename,Total Pembayaran,product_categories,11,506389.60,0,0,49350222.60,21280.000,celengan,data/interim/cleaned_separate/ecommerce_2024_1...
4,ecommerce_2025_02_february,E-Commerce Sales - February 2025,ecommerce_sales,957,957,0,100.00,source_column,Total Pembayaran,product_categories,9,684076.10,0,0,48015557.90,20998.000,celengan,data/interim/cleaned_separate/ecommerce_2025_0...
5,ecommerce_2025_07_july,E-Commerce Sales - July 2025,ecommerce_sales,766,766,0,100.00,dataset_period_from_filename,Total Pembayaran,product_categories,7,732356.00,0,0,34504162.00,19500.000,aksesoris_pintu,data/interim/cleaned_separate/ecommerce_2025_0...
6,ecommerce_2025_11_november,E-Commerce Sales - November 2025,ecommerce_sales,1131,1131,0,100.00,source_column,Total Pembayaran,product_categories,11,405037.90,0,0,40747696.90,18900.000,mangkok_sambal_saus,data/interim/cleaned_separate/ecommerce_2025_1...
7,personal_finance,Personal Finance Dataset,personal_finance,1500,1500,0,100.00,source_column,Amount,Category,15,4718.09,3000,0,1958971.72,1156.285,rent,data/interim/cleaned_separate/personal_finance...


## Validasi Hasil Cleaning Tiap Dataset

In [5]:
validation_rows = []

for dataset_id, frame in cleaned_datasets.items():
    validation_rows.append(
        {
            'dataset_id': dataset_id,
            'records': len(frame),
            'missing_date': int(frame['date'].isna().sum()),
            'missing_amount': int(frame['amount'].isna().sum()),
            'negative_amount': int((frame['amount'] < 0).sum()),
            'duplicate_keys': int(frame.duplicated(subset=['date', 'amount', 'category', 'description', 'transaction_type']).sum()),
            'unique_categories': int(frame['category'].nunique()),
            'date_min': frame['date'].min(),
            'date_max': frame['date'].max(),
        }
    )

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

,dataset_id,records,missing_date,missing_amount,negative_amount,duplicate_keys,unique_categories,date_min,date_max
0,ecommerce_2024_01_january,431,0,0,0,0,51,2024-01-01 00:05:00,2024-01-31 22:56:00
1,ecommerce_2024_06_june,697,0,0,0,0,55,2024-06-01 03:22:00,2024-06-30 23:36:00
2,ecommerce_2024_12_december,1214,0,0,0,0,97,2024-12-01 00:00:00,2024-12-01 00:00:00
3,ecommerce_2025_02_february,957,0,0,0,0,87,2025-02-01 00:40:00,2025-02-28 23:23:00
4,ecommerce_2025_07_july,766,0,0,0,0,68,2025-07-01 00:00:00,2025-07-01 00:00:00
5,ecommerce_2025_11_november,1131,0,0,0,0,62,2025-11-01 06:08:00,2025-11-30 23:17:00
6,daily_household_transactions,2445,0,0,0,0,50,2015-01-01 00:00:00,2018-09-20 12:04:08
7,personal_finance,1500,0,0,0,0,10,2020-01-02 00:00:00,2024-12-29 00:00:00


## Save Cleaning Report

In [6]:
def build_cleaning_text_report(reports):
    lines = [
        '=' * 100,
        'DATA CLEANING REPORT - 8 DATASET TERPISAH',
        '=' * 100,
        '',
    ]

    for dataset_id, report in reports.items():
        lines.extend(
            [
                f'DATASET: {dataset_id}',
                '-' * 100,
                f"Name            : {report['dataset_name']}",
                f"Initial Records : {report['initial_records']:,}",
                f"Final Records   : {report['final_records']:,}",
                f"Retention       : {report['retention_pct']}%",
                f"Removed Records : {report['removed_records']:,}",
                f"Date Source     : {report['date_source']}",
                f"Capped Records  : {report['capped_records']:,}",
                f"Output          : {report['output_path']}",
                'Cleaning Steps:',
            ]
        )
        for step in report['cleaning_steps']:
            lines.append(
                f"- {step['step']}: {step['before_records']:,} -> {step['after_records']:,} "
                f"(removed {step['removed_records']:,})"
            )
        lines.append('')

    lines.append('=' * 100)
    return '\n'.join(lines)

cleaning_json_path = interim_data_path / 'cleaning_reports_by_dataset.json'
cleaning_summary_path = interim_data_path / 'cleaning_summary_by_dataset.csv'
text_report_path = reports_path / '03_cleaning_report_by_dataset.txt'

with open(cleaning_json_path, 'w', encoding='utf-8') as file:
    json.dump(to_python(cleaning_reports), file, indent=2)
cleaning_summary_df.to_csv(cleaning_summary_path, index=False)
text_report_path.write_text(build_cleaning_text_report(cleaning_reports), encoding='utf-8')

print('Cleaning outputs saved:')
print(f'- {cleaning_json_path.relative_to(project_root)}')
print(f'- {cleaning_summary_path.relative_to(project_root)}')
print(f'- {text_report_path.relative_to(project_root)}')

Cleaning outputs saved:
- data/interim/cleaning_reports_by_dataset.json
- data/interim/cleaning_summary_by_dataset.csv
- reports/03_cleaning_report_by_dataset.txt


## Visualisasi Akhir Dataset Cleaned
Visualisasi di bawah memakai summary dan agregasi per dataset. Ini bukan export dataset transaksi gabungan.

In [7]:
from html import escape


def build_bar_chart_html(frame, label_col, value_col, title, color, value_suffix='', value_format=',.2f'):
    chart = frame[[label_col, value_col]].copy()
    chart[value_col] = pd.to_numeric(chart[value_col], errors='coerce').fillna(0)
    max_value = chart[value_col].max() or 1
    rows = []

    for _, row in chart.iterrows():
        label = escape(str(row[label_col]))
        value = float(row[value_col])
        width = max(2, value / max_value * 100)
        value_text = format(value, value_format) + value_suffix
        rows.append(
            f'''<div class="bar-row">
                <div class="bar-label">{label}</div>
                <div class="bar-track"><div class="bar-fill" style="width:{width:.2f}%; background:{color};"></div></div>
                <div class="bar-value">{value_text}</div>
            </div>'''
        )

    return f'''<section class="chart-card">
        <h3>{escape(title)}</h3>
        {''.join(rows)}
    </section>'''


plot_df = cleaning_summary_df.sort_values('final_records', ascending=True).copy()
plot_df['short_name'] = plot_df['dataset_id'].str.replace('ecommerce_', 'ecom_', regex=False)

html = f'''
<style>
.dashboard-grid {{ display: grid; grid-template-columns: repeat(2, minmax(300px, 1fr)); gap: 16px; font-family: Arial, sans-serif; }}
.chart-card {{ border: 1px solid #d8dee4; border-radius: 8px; padding: 14px; background: #ffffff; }}
.chart-card h3 {{ margin: 0 0 12px 0; font-size: 16px; color: #1f2937; }}
.bar-row {{ display: grid; grid-template-columns: minmax(140px, 1.5fr) 2fr minmax(90px, .9fr); gap: 8px; align-items: center; margin: 8px 0; }}
.bar-label {{ font-size: 12px; color: #374151; overflow-wrap: anywhere; }}
.bar-track {{ height: 14px; background: #eef2f7; border-radius: 999px; overflow: hidden; }}
.bar-fill {{ height: 100%; border-radius: 999px; }}
.bar-value {{ font-size: 12px; color: #111827; text-align: right; font-variant-numeric: tabular-nums; }}
@media (max-width: 900px) {{ .dashboard-grid {{ grid-template-columns: 1fr; }} }}
</style>
<div class="dashboard-grid">
{build_bar_chart_html(plot_df, 'short_name', 'final_records', 'Final Records Setelah Cleaning', '#2f6f73', value_format=',.0f')}
{build_bar_chart_html(plot_df, 'short_name', 'retention_pct', 'Retention Rate per Dataset', '#6f5f90', '%')}
{build_bar_chart_html(plot_df, 'short_name', 'total_amount', 'Total Amount Setelah Cleaning', '#bf7b45', value_format=',.0f')}
{build_bar_chart_html(plot_df, 'short_name', 'median_amount', 'Median Amount Setelah Cleaning', '#3f7f5f', value_format=',.0f')}
</div>
'''

dashboard_path = figures_path / '03_cleaning_summary_dashboard.html'
dashboard_path.write_text(html, encoding='utf-8')
display(HTML(html))
print(f'Dashboard saved: {dashboard_path.relative_to(project_root)}')


Dashboard saved: reports/figures/03_cleaning_summary_dashboard.html


## Visualisasi Kategori Teratas
Grafik ini melihat kategori bernilai transaksi terbesar dari hasil cleaning masing-masing dataset.

In [8]:
from html import escape

category_rows = []

for dataset_id, frame in cleaned_datasets.items():
    category_amount = frame.groupby('category', dropna=False)['amount'].sum().sort_values(ascending=False).head(5)
    for category, amount in category_amount.items():
        category_rows.append(
            {
                'dataset_id': dataset_id,
                'category': category,
                'amount': float(amount),
                'label': f"{dataset_id.replace('ecommerce_', 'ecom_')} | {category}",
            }
        )

category_summary_df = pd.DataFrame(category_rows).sort_values('amount', ascending=False).head(20)
display(category_summary_df)

max_amount = category_summary_df['amount'].max() if not category_summary_df.empty else 1
rows = []
for _, row in category_summary_df.sort_values('amount').iterrows():
    width = max(2, float(row['amount']) / max_amount * 100) if max_amount else 2
    rows.append(
        f'''<div class="bar-row">
            <div class="bar-label">{escape(str(row['label']))}</div>
            <div class="bar-track"><div class="bar-fill" style="width:{width:.2f}%;"></div></div>
            <div class="bar-value">{float(row['amount']):,.0f}</div>
        </div>'''
    )

html = f'''
<style>
.chart-card {{ border: 1px solid #d8dee4; border-radius: 8px; padding: 14px; background: #ffffff; font-family: Arial, sans-serif; }}
.chart-card h3 {{ margin: 0 0 12px 0; font-size: 16px; color: #1f2937; }}
.bar-row {{ display: grid; grid-template-columns: minmax(260px, 1.7fr) 3fr minmax(100px, .8fr); gap: 8px; align-items: center; margin: 8px 0; }}
.bar-label {{ font-size: 12px; color: #374151; overflow-wrap: anywhere; }}
.bar-track {{ height: 14px; background: #eef2f7; border-radius: 999px; overflow: hidden; }}
.bar-fill {{ height: 100%; border-radius: 999px; background: #3f7f5f; }}
.bar-value {{ font-size: 12px; color: #111827; text-align: right; font-variant-numeric: tabular-nums; }}
</style>
<section class="chart-card">
<h3>Top 20 Kategori berdasarkan Total Amount Cleaned</h3>
{''.join(rows)}
</section>
'''

dashboard_path = figures_path / '03_top_categories_cleaned.html'
dashboard_path.write_text(html, encoding='utf-8')
display(HTML(html))
print(f'Dashboard saved: {dashboard_path.relative_to(project_root)}')


,dataset_id,category,amount,label
25,ecommerce_2025_11_november,mangkok_sambal_saus,14120014.0,ecom_2025_11_november | mangkok_sambal_saus
15,ecommerce_2025_02_february,seal_baut_roof,12519695.7,ecom_2025_02_february | seal_baut_roof
20,ecommerce_2025_07_july,seal_baut_roof,11332678.0,ecom_2025_07_july | seal_baut_roof
10,ecommerce_2024_12_december,seal_baut_roof,9799317.0,ecom_2024_12_december | seal_baut_roof
11,ecommerce_2024_12_december,celengan,9196765.0,ecom_2024_12_december | celengan
26,ecommerce_2025_11_november,seal_baut_roof,7591583.3,ecom_2025_11_november | seal_baut_roof
12,ecommerce_2024_12_december,mangkok_sambal_saus,7382271.0,ecom_2024_12_december | mangkok_sambal_saus
21,ecommerce_2025_07_july,nampan_tray,7006939.0,ecom_2025_07_july | nampan_tray
5,ecommerce_2024_06_june,celengan,6267305.0,ecom_2024_06_june | celengan
27,ecommerce_2025_11_november,celengan,5474575.0,ecom_2025_11_november | celengan


Dashboard saved: reports/figures/03_top_categories_cleaned.html


## Output Cleaning
Cleaning selesai. Semua dataset tetap tersimpan terpisah di `data/interim/cleaned_separate/`, dan visualisasi memakai summary/agregasi agar perbandingan antar dataset tetap mudah dipahami tanpa membuat satu file full gabungan.